# 🛡️ LoanGuard — Model Training & Fine-Tuning Notebook

This notebook covers:
1. Load Preprocessed Data
2. Baseline Model (Logistic Regression)
3. XGBoost — Initial Training
4. Handling Class Imbalance
5. Hyperparameter Tuning (Optuna)
6. Cross Validation
7. Final Model Evaluation
8. Feature Importance Analysis
9. SHAP Explainability
10. MLflow Experiment Tracking
11. Save Best Model

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Explainability
import shap

# Experiment tracking
import mlflow
import mlflow.xgboost

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ All libraries loaded')
print(f'XGBoost version: {xgb.__version__}')
print(f'MLflow version: {mlflow.__version__}')

## 2. Load Preprocessed Data

In [ ]:
# Load train/test splits from EDA notebook
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'Train default rate: {y_train.mean():.3f}')
print(f'Test default rate:  {y_test.mean():.3f}')
print(f'\nFeatures: {X_train.columns.tolist()}')

In [ ]:
# Class imbalance ratio — needed for scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'Negative samples: {neg:,}')
print(f'Positive samples: {pos:,}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

## 3. Evaluation Helper Functions

In [ ]:
def evaluate_model(model, X_test, y_test, model_name='Model'):
    """Full evaluation of a trained model"""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_pred_proba)
    ap  = average_precision_score(y_test, y_pred_proba)

    print(f'\n{"="*50}')
    print(f'  {model_name} Evaluation')
    print(f'{"="*50}')
    print(f'  ROC-AUC:          {auc:.4f}')
    print(f'  Avg Precision:    {ap:.4f}')
    print(f'\n{classification_report(y_test, y_pred)}')

    return auc, ap, y_pred_proba


def plot_roc_curves(models_dict, X_test, y_test):
    """Plot ROC curves for multiple models"""
    plt.figure(figsize=(8, 6))

    for name, model in models_dict.items():
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {auc:.4f})')

    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('../notebooks/roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix(model, X_test, y_test, model_name='Model'):
    """Plot confusion matrix"""
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Default', 'Default'],
                yticklabels=['Non-Default', 'Default'], ax=axes[0])
    axes[0].set_title(f'{model_name} — Confusion Matrix (Counts)')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')

    sns.heatmap(cm_pct, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=['Non-Default', 'Default'],
                yticklabels=['Non-Default', 'Default'], ax=axes[1])
    axes[1].set_title(f'{model_name} — Confusion Matrix (Percentages)')
    axes[1].set_ylabel('Actual')
    axes[1].set_xlabel('Predicted')

    plt.tight_layout()
    plt.savefig('../notebooks/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()


print('✅ Evaluation helpers ready')

## 4. Baseline Models

In [ ]:
# Logistic Regression baseline
print('Training Logistic Regression baseline...')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)

lr_auc, lr_ap, _ = evaluate_model(lr, X_test_scaled, y_test, 'Logistic Regression')

In [ ]:
# XGBoost baseline — default params
print('Training XGBoost baseline...')
xgb_base = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc',
    verbosity=0
)
xgb_base.fit(X_train, y_train)

xgb_base_auc, xgb_base_ap, _ = evaluate_model(xgb_base, X_test, y_test, 'XGBoost Baseline')

In [ ]:
# Baseline comparison
baseline_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost Baseline'],
    'ROC-AUC': [lr_auc, xgb_base_auc],
    'Avg Precision': [lr_ap, xgb_base_ap]
})

print(baseline_results.to_string(index=False))
print(f'\nXGBoost improvement over LR: +{(xgb_base_auc - lr_auc):.4f} AUC')

## 5. Handling Class Imbalance — Comparison

In [ ]:
# Strategy 1 — scale_pos_weight (already used above)
# Strategy 2 — SMOTE oversampling
print('Applying SMOTE oversampling...')
smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_smote, y_train_smote = smote.fit_resample(
    X_train.sample(50000, random_state=42),
    y_train.loc[X_train.sample(50000, random_state=42).index]
)

print(f'Before SMOTE: {(y_train[:50000]==1).sum()} positives')
print(f'After SMOTE:  {(y_train_smote==1).sum()} positives')

xgb_smote = xgb.XGBClassifier(
    n_estimators=100, random_state=42,
    eval_metric='auc', verbosity=0
)
xgb_smote.fit(X_train_smote, y_train_smote)
smote_auc, smote_ap, _ = evaluate_model(xgb_smote, X_test, y_test, 'XGBoost + SMOTE')

In [ ]:
# Imbalance strategy comparison
imbalance_results = pd.DataFrame({
    'Strategy': ['scale_pos_weight', 'SMOTE'],
    'ROC-AUC': [xgb_base_auc, smote_auc],
    'Avg Precision': [xgb_base_ap, smote_ap]
})
print(imbalance_results.to_string(index=False))
print('\n💡 We use scale_pos_weight — faster and avoids synthetic data issues')

## 6. Hyperparameter Tuning with Optuna

In [ ]:
# Use a sample for faster tuning
TUNE_SAMPLE = 80000
X_tune = X_train.sample(TUNE_SAMPLE, random_state=42)
y_tune = y_train.loc[X_tune.index]

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 2.0),
        'scale_pos_weight':  scale_pos_weight,
        'random_state':      42,
        'eval_metric':       'auc',
        'verbosity':         0
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    model = xgb.XGBClassifier(**params)

    scores = cross_val_score(
        model, X_tune, y_tune,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()

print(f'Running Optuna hyperparameter tuning on {TUNE_SAMPLE:,} samples...')
print('This will take 5-10 minutes...\n')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\n✅ Tuning complete!')
print(f'Best AUC (CV): {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Optuna optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial history
trial_values = [t.value for t in study.trials]
best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

axes[0].plot(trial_values, alpha=0.5, color='#3498db', label='Trial AUC')
axes[0].plot(best_so_far, color='#e74c3c', linewidth=2, label='Best AUC so far')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Optuna Optimization History', fontsize=13, fontweight='bold')
axes[0].legend()

# Param importance
importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()),
             color='#2ecc71', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/optuna_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Train Final Model with Best Params

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'eval_metric': 'auc',
    'verbosity': 0
})

print('Training final model with best hyperparameters...')
print(f'Using full training set: {X_train.shape[0]:,} samples\n')

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

final_auc, final_ap, y_pred_proba = evaluate_model(
    final_model, X_test, y_test, 'Final Tuned XGBoost'
)
print(f'\n📈 Improvement over baseline: +{(final_auc - xgb_base_auc):.4f} AUC')

## 8. Cross Validation

In [ ]:
print('Running 5-fold Stratified Cross Validation...')
print('(Using 100K sample for speed)\n')

cv_sample = X_train.sample(100000, random_state=42)
cv_labels  = y_train.loc[cv_sample.index]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = xgb.XGBClassifier(**best_params)

cv_scores = cross_val_score(
    cv_model, cv_sample, cv_labels,
    cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
)

print(f'\nCV Scores: {cv_scores}')
print(f'Mean AUC:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Min AUC:   {cv_scores.min():.4f}')
print(f'Max AUC:   {cv_scores.max():.4f}')

In [ ]:
# Plot CV scores
plt.figure(figsize=(8, 4))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
colors = ['#2ecc71' if s >= cv_scores.mean() else '#e74c3c' for s in cv_scores]

plt.bar(folds, cv_scores, color=colors, edgecolor='black', alpha=0.8)
plt.axhline(y=cv_scores.mean(), color='blue', linestyle='--',
            linewidth=2, label=f'Mean = {cv_scores.mean():.4f}')
plt.axhline(y=cv_scores.mean() - cv_scores.std(), color='orange',
            linestyle=':', label=f'±1 std = {cv_scores.std():.4f}')
plt.axhline(y=cv_scores.mean() + cv_scores.std(), color='orange', linestyle=':')

plt.ylim(cv_scores.min() - 0.01, cv_scores.max() + 0.01)
plt.ylabel('ROC-AUC')
plt.title('5-Fold Cross Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
for i, v in enumerate(cv_scores):
    plt.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../notebooks/cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Full Model Evaluation

In [ ]:
# ROC curve comparison — all models
models_to_compare = {
    'Logistic Regression': lr,
    'XGBoost Baseline': xgb_base,
    'XGBoost Tuned': final_model
}

plt.figure(figsize=(8, 6))
for name, model in models_to_compare.items():
    if name == 'Logistic Regression':
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../notebooks/roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall curve
plt.figure(figsize=(8, 6))

for name, model in models_to_compare.items():
    if name == 'Logistic Regression':
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]

    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    plt.plot(recall, precision, linewidth=2, label=f'{name} (AP = {ap:.4f})')

plt.axhline(y=y_test.mean(), color='gray', linestyle='--',
            label=f'Baseline (AP = {y_test.mean():.4f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../notebooks/precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix for final model
plot_confusion_matrix(final_model, X_test, y_test, 'Final Tuned XGBoost')

In [ ]:
# Prediction probability distribution
plt.figure(figsize=(10, 5))

plt.hist(y_pred_proba[y_test == 0], bins=50, alpha=0.7,
         color='#2ecc71', label='Non-Default', density=True)
plt.hist(y_pred_proba[y_test == 1], bins=50, alpha=0.7,
         color='#e74c3c', label='Default', density=True)
plt.axvline(x=0.3, color='orange', linestyle='--', label='Low/Med threshold (0.3)')
plt.axvline(x=0.6, color='red', linestyle='--', label='Med/High threshold (0.6)')
plt.xlabel('Predicted Default Probability', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Prediction Probability Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../notebooks/probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Feature Importance Analysis

In [ ]:
# XGBoost native feature importance — 3 types
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

importance_types = ['weight', 'gain', 'cover']
titles = ['Frequency (Weight)', 'Gain', 'Cover']
colors = ['#3498db', '#2ecc71', '#e74c3c']

for ax, imp_type, title, color in zip(axes, importance_types, titles, colors):
    importance = final_model.get_booster().get_score(importance_type=imp_type)
    importance_df = pd.Series(importance).sort_values(ascending=True).tail(15)

    ax.barh(importance_df.index, importance_df.values,
            color=color, edgecolor='black', alpha=0.8)
    ax.set_title(f'Feature Importance ({title})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Score')

plt.suptitle('XGBoost Feature Importance — Three Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/feature_importance_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. SHAP Explainability

In [ ]:
print('Computing SHAP values (using 5000 samples for speed)...')
X_shap = X_test.sample(5000, random_state=42)

explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

print('✅ SHAP values computed')

In [ ]:
# SHAP summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='bar',
                  show=False, max_display=15)
plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm plot — shows direction of impact
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Red = high feature value, Blue = low feature value')
print('💡 Right of center = increases default risk')

In [ ]:
# SHAP waterfall for a single prediction — explain one decision
idx = 0
shap_explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_shap.iloc[idx],
    feature_names=X_shap.columns.tolist()
)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation, max_display=12, show=False)
plt.title('SHAP Waterfall — Single Prediction Explanation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Log Everything to MLflow

In [ ]:
mlflow.set_experiment('loanguard-credit-risk')

with mlflow.start_run(run_name='final_tuned_xgboost_notebook'):

    # Log best params
    mlflow.log_params(best_params)

    # Log metrics
    mlflow.log_metric('roc_auc',          final_auc)
    mlflow.log_metric('avg_precision',    final_ap)
    mlflow.log_metric('cv_mean_auc',      cv_scores.mean())
    mlflow.log_metric('cv_std_auc',       cv_scores.std())
    mlflow.log_metric('baseline_auc',     xgb_base_auc)
    mlflow.log_metric('lr_baseline_auc',  lr_auc)

    # Log model
    mlflow.xgboost.log_model(final_model, 'model')

    # Log plots as artifacts
    plots = [
        '../notebooks/roc_comparison.png',
        '../notebooks/confusion_matrix.png',
        '../notebooks/shap_importance.png',
        '../notebooks/shap_beeswarm.png',
        '../notebooks/cv_scores.png',
        '../notebooks/optuna_results.png',
        '../notebooks/probability_distribution.png'
    ]
    for plot in plots:
        try:
            mlflow.log_artifact(plot)
        except:
            pass

    mlflow.set_tag('tuning_method', 'optuna')
    mlflow.set_tag('cv_folds', '5')
    mlflow.set_tag('promoted', 'true')

    run_id = mlflow.active_run().info.run_id

print(f'✅ Run logged to MLflow')
print(f'   Run ID: {run_id}')
print(f'   View at: http://localhost:5000')

## 13. Save Final Model

In [ ]:
import os
import json

os.makedirs('../models', exist_ok=True)

# Save model
final_model.save_model('../models/best_model.json')
print('✅ Model saved to models/best_model.json')

# Save model metadata
metadata = {
    'roc_auc':         round(final_auc, 4),
    'avg_precision':   round(final_ap, 4),
    'cv_mean_auc':     round(float(cv_scores.mean()), 4),
    'cv_std_auc':      round(float(cv_scores.std()), 4),
    'n_features':      len(X_train.columns),
    'train_samples':   len(X_train),
    'test_samples':    len(X_test),
    'tuning_trials':   30,
    'best_params':     {k: v for k, v in best_params.items()
                        if k not in ['random_state', 'eval_metric', 'verbosity']},
    'mlflow_run_id':   run_id
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Metadata saved to models/model_metadata.json')
print()
print(json.dumps(metadata, indent=2))

## 14. Final Summary

In [ ]:
print('=' * 58)
print('     LOANGUARD — MODEL TRAINING SUMMARY')
print('=' * 58)
print()
print('Model Comparison:')
print(f'  Logistic Regression:   AUC = {lr_auc:.4f}')
print(f'  XGBoost Baseline:      AUC = {xgb_base_auc:.4f}')
print(f'  XGBoost Tuned (final): AUC = {final_auc:.4f}  ✅ BEST')
print()
print('Cross Validation (5-fold):')
print(f'  Mean AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print('Tuning:')
print(f'  Method:  Optuna (30 trials)')
print(f'  Best CV AUC: {study.best_value:.4f}')
print()
print('Top 3 most important features (SHAP):')
shap_mean = np.abs(shap_values).mean(axis=0)
top_features = pd.Series(shap_mean, index=X_shap.columns).sort_values(ascending=False).head(3)
for feat, val in top_features.items():
    print(f'  {feat}: {val:.4f}')
print()
print('Artifacts saved:')
print('  models/best_model.json')
print('  models/model_metadata.json')
print('  MLflow run logged')
print()
print('✅ Ready for API deployment!')
print('=' * 58)